In [0]:
from pyspark.sql.functions import sha2, concat_ws, col

def check_row_integrity():
    # Define columns that shouldn't change between Bronze and Silver
    core_cols = ["user_id", "event_type", "product_id", "category_id", "price"]
    
    # Create hash for Bronze
    bronze_hashed = spark.table("workspace.dev_bronze_layer.events_raw") \
        .withColumn("row_hash", sha2(concat_ws("||", *core_cols), 256))
        
    # Create hash for Silver
    silver_hashed = spark.table("workspace.dev_silver_layer.events_cleaned") \
        .withColumn("row_hash", sha2(concat_ws("||", *core_cols), 256))

    # Test: Pick a random row from Silver and ensure the hash exists in Bronze
    sample_hash = silver_hashed.select("row_hash").first()[0]
    exists_in_bronze = bronze_hashed.filter(col("row_hash") == sample_hash).count() > 0
    
    assert exists_in_bronze, "ERROR: Row-level integrity failed. Silver row hash not found in Bronze!"
    print("Row-level integrity verified via SHA256.")

check_row_integrity()
